In [4]:
import pandas as pd

df = pd.read_csv("data/image/HAM10000_metadata.csv")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'image/data/HAM10000_metadata.csv'

In [ ]:
import os
import torch
from torch import nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [ ]:
class SkinCancerDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

        # Map the 7 labels to binary (0 = no_cancer, 1 = cancer)
        self.label_map = {
            "mel": 1,   # Melanoma
            "bcc": 1,   # Basal Cell Carcinoma
            "akiec": 1, # Bowen’s / SCC in situ
            "nv": 0,
            "bkl": 0,
            "df": 0,
            "vasc": 0
        }

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_name = row['image_id'] + ".jpg"  # Change if png/jpeg differs
        img_path = os.path.join(self.img_dir, img_name)

        image = Image.open(img_path).convert("RGB")
        label = self.label_map[row['dx']]

        if self.transform:
            image = self.transform(image)

        return image, label



In [ ]:

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

dataset = SkinCancerDataset(
    csv_file="data/HAM10000_metadata.csv",
    img_dir="data/ham10000_images_part_1/",  # path to folder with .jpg images
    transform=transform
)

In [2]:
dataset

NameError: name 'dataset' is not defined

In [3]:
BATCH_SIZE =32

from torch.utils.data import DataLoader
train_dataloader = DataLoader(
    dataset=dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

NameError: name 'dataset' is not defined

In [7]:
import torchvision

def get_vit_model(num_classes=2):
    # Load pretrained ViT
    model = torchvision.models.vit_b_16(weights="IMAGENET1K_V1")  # Pretrained on ImageNet

    # Replace the head for binary classification
    in_features = model.heads.head.in_features
    model.heads.head = nn.Linear(in_features, num_classes)

    return model

model = get_vit_model(num_classes=2)

In [ ]:
from tqdm.auto import tqdm
EPOCHS = 5
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


for epoch in tqdm(range(EPOCHS)):
    train_loss = 0
    model.train()
    for batch in train_dataloader:
        inputs, labels = batch
        inputs, labels = inputs, labels
        output = model(inputs)
        loss = loss_fn(output, labels)
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(
        f"Epoch {epoch}, train loss {train_loss / len(train_dataloader)}"
    )

MODEL_PATH = Path("models")
MODEL_NAME = "model.pth"
MODEL_SAVE_PATH = MODEL_PATH / MODEL_NAME


MODEL_PATH.mkdir(parents=True, exist_ok=True)
# Save
torch.save(obj=model.state_dict(), f=MODEL_SAVE_PATH)

  0%|          | 0/5 [00:00<?, ?it/s]

In [1]:
import pandas as pd

df = pd.read_csv("data/image/HAM10000_metadata.csv")

In [2]:
df.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear
